In [1]:
# COMPLETE SETUP CELL - Run this FIRST

from google.colab import drive
import os

# Mount Drive (skip if already mounted)
try:
    drive.mount('/content/drive', force_remount=False)
except:
    pass  # Already mounted

# Navigate to project folder
# Update this path to match your project location in Google Drive
project_path = '/content/drive/MyDrive/DLP_project'  # Change this to your actual path
os.chdir(project_path)

# Change to notebooks folder (so ../results/ and ../data/ paths work)
os.chdir('notebooks')

print(f"Current directory: {os.getcwd()}")
print(f"Can access data/processed: {os.path.exists('../data/processed')}")
print(f"Can access results: {os.path.exists('../results')}")
print(f"Can access models: {os.path.exists('../models')}")


Mounted at /content/drive
Current directory: /content/drive/MyDrive/DLP_project/notebooks
Can access data/processed: True
Can access results: True
Can access models: True


# Text Generation

This notebook loads all trained models and generates Urdu poetry using specified seed words.


In [2]:
# Install required libraries
!pip install -q tensorflow numpy pandas


In [3]:
import os
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Set random seeds for reproducibility
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

# Create directories
os.makedirs('../results/generated_poems', exist_ok=True)


## Load Tokenizer and Metadata


In [4]:
# Load tokenizer and metadata
with open('../data/processed/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

with open('../data/processed/metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

sequence_length = metadata['sequence_length']
vocab_size = metadata['vocab_size']

print(f"Sequence length: {sequence_length}")
print(f"Vocabulary size: {vocab_size}")


Sequence length: 20
Vocabulary size: 10505


## Text Generation Function


In [5]:
def generate_text(model, seed_text, tokenizer, sequence_length, num_words=50, temperature=1.0):
    """Generate text given a seed text"""
    generated_text = seed_text
    input_sequence = tokenizer.texts_to_sequences([seed_text])[0]

    for _ in range(num_words):
        # Pad sequence
        padded_sequence = pad_sequences([input_sequence], maxlen=sequence_length, padding='pre', truncating='pre')

        # Predict next word
        predictions = model.predict(padded_sequence, verbose=0)[0]

        # Apply temperature
        predictions = np.log(predictions + 1e-8) / temperature
        predictions = np.exp(predictions)
        predictions = predictions / np.sum(predictions)

        # Sample next word
        next_token = np.random.choice(len(predictions), p=predictions)

        # Add to sequence
        input_sequence.append(next_token)
        input_sequence = input_sequence[-sequence_length:]

        # Convert token to word
        word = tokenizer.index_word.get(next_token, '<OOV>')
        generated_text += ' ' + word

    return generated_text

def generate_poetry_lines(model, seed_word, tokenizer, sequence_length, num_lines=5, words_per_line=10, temperature=1.0):
    """Generate multiple lines of poetry"""
    lines = []
    current_seed = seed_word

    for i in range(num_lines):
        # Generate a line
        line = generate_text(model, current_seed, tokenizer, sequence_length,
                           num_words=words_per_line, temperature=temperature)
        lines.append(line)
        # Use last few words as seed for next line
        words = line.split()
        if len(words) >= 3:
            current_seed = ' '.join(words[-3:])
        else:
            current_seed = seed_word

    return lines

def generate_multiple_samples(model, seed_word, tokenizer, sequence_length, temperatures=[0.7, 1.0, 1.3], samples_per_temp=5):
    """Generate multiple samples with different temperatures as per project requirements"""
    all_samples = {}

    for temp in temperatures:
        temp_samples = []
        for _ in range(samples_per_temp):
            lines = generate_poetry_lines(model, seed_word, tokenizer, sequence_length,
                                         num_lines=5, words_per_line=10, temperature=temp)
            temp_samples.append('\n'.join(lines))
        all_samples[temp] = temp_samples

    return all_samples

print("Generation functions defined!")


Generation functions defined!


In [6]:
# Seed words as specified
seed_words = ['محبت', 'دل', 'شام', 'یاد', 'خوشی']

print("Seed words:")
for word in seed_words:
    print(f"  - {word}")


Seed words:
  - محبت
  - دل
  - شام
  - یاد
  - خوشی


## Load All Models and Generate Poetry


In [7]:
# Define all model-optimizer combinations
models_config = [
    ('RNN', 'Adam', '../models/rnn_adam.h5'),
    ('RNN', 'RMSprop', '../models/rnn_rmsprop.h5'),
    ('RNN', 'SGD', '../models/rnn_sgd.h5'),
    ('LSTM', 'Adam', '../models/lstm_adam.h5'),
    ('LSTM', 'RMSprop', '../models/lstm_rmsprop.h5'),
    ('LSTM', 'SGD', '../models/lstm_sgd.h5'),
    ('Transformer', 'Adam', '../models/transformer_adam.h5'),
    ('Transformer', 'RMSprop', '../models/transformer_rmsprop.h5'),
    ('Transformer', 'SGD', '../models/transformer_sgd.h5'),
]

# Load models and generate poetry
all_generated_poems = {}

for model_name, optimizer_name, model_path in models_config:
    print(f"\n{'='*60}")
    print(f"Loading {model_name} - {optimizer_name}")
    print(f"{'='*60}")

    try:
        model = load_model(model_path)
        model_key = f"{model_name}_{optimizer_name}"
        all_generated_poems[model_key] = {}

        # Generate poetry for each seed word with multiple temperatures
        for seed_word in seed_words:
            print(f"\nGenerating poetry with seed: {seed_word}")
            temperatures = [0.7, 1.0, 1.3]  # As per project requirements
            samples = generate_multiple_samples(model, seed_word, tokenizer, sequence_length,
                                               temperatures=temperatures, samples_per_temp=5)
            all_generated_poems[model_key][seed_word] = samples

            # Display sample generations
            print(f"\nSample generations (showing first sample per temperature):")
            for temp in temperatures:
                print(f"\n  Temperature {temp} (Sample 1 of 5):")
                print(f"  {samples[temp][0][:200]}...")  # Show first 200 chars

    except Exception as e:
        print(f"Error loading model {model_path}: {e}")
        print("Skipping this model...")

print("\n\nAll poetry generation completed!")



Loading RNN - Adam



Generating poetry with seed: محبت

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  محبت سب راج دیکھیے دیا ہے مجھے مجھے اندیش قدم ہوگا
اندیش قدم ہوگا کیا نگار پر بھی ہیں ہم کو غم تھا مجھے
غم تھا مجھے رب آج اسے لیکن زلف لکھنؤ ہے ساقی میں ہے
ساقی میں ہے چاند تو سے ہنوز درخشاں اپنا آج ب...

  Temperature 1.0 (Sample 1 of 5):
  محبت میرا سحر کائی ہے مجھے آوارگی بوسہ ہستی آنکھیں عشق
ہستی آنکھیں عشق میرا تبرک کی راہ آشیانہ جالبؔ سزا شمار تھا کوئی
شمار تھا کوئی بالا پہچان آوے ہے آئنہ خانے سوں مان کھلا کہوں
مان کھلا کہوں چار سوا...

  Temperature 1.3 (Sample 1 of 5):
  محبت چلتی جلوہ کرتے جاؤ سی نہیں سجتے پرکاری سامعین کدہ
پرکاری سامعین کدہ گوارا گلہ ناصحو چاہے ہے ہم کون اٹھائیے پاڑ جائے
اٹھائیے پاڑ جائے ہوگی قسم گھڑے بیٹھیں سجتے خانے سخن ہستی سے ہوجائیے
ہستی سے ہوج...

Generating poetry with seed: دل

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  دل کا بتیاں کریں ہم سے آ گئے گے جادو بہتا
گے جا


Generating poetry with seed: محبت

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  محبت پکارنے ابھرتے لگا پہنچا کیا ہے مجھے مجھے ہوں میں
مجھے ہوں میں ہوں میں لیکن کیا کیا کیا ہوا ہے مجھے بھی
ہے مجھے بھی رنگ آوے ہے ساقی کا پھری آخر ہے ساقی میرا
ہے ساقی میرا چاند ہے ہے موجود پر بھی نہ...

  Temperature 1.0 (Sample 1 of 5):
  محبت مژدہ بجان تاختن آئے ہیں ہم تک تنہائی گے ہم
تنہائی گے ہم کو فریب ساعت ہے سراپا میں کیفیت دھجی بھی نہیں
دھجی بھی نہیں سکتا ہے آن کھلانے کا نخوت ہے ابھی نثار نام
ابھی نثار نام معتبر کی ہے شاہاں بھی ...

  Temperature 1.3 (Sample 1 of 5):
  محبت روپ رہا دیکھنا بیداری جائے نہاں آنگن، تلک نگاہی کرتا
تلک نگاہی کرتا ہوں بخشیدن دراز حال طاری ہے تعارف میں پرکھا برفیلی
میں پرکھا برفیلی دیجے چھل دے آوے ہے ٹھکرا مانگے ہے ہمیں بھولنے
ہے ہمیں بھولن...

Generating poetry with seed: دل

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  دل خلش تاثیر لیا ہے بہت میں ہوں میں خیال شوخ
میں


Generating poetry with seed: محبت

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  محبت سے تھا ہے ہے سے ماروں ہے ہے ہیں دھند
ہے ہیں دھند تھا معدود ہے ہے ہیں تھا ہے ہے کیا ہے
ہے کیا ہے ہے میں سے نہیں ہے ہیں کو سے ہیں ہے
سے ہیں ہے ہے ہے ہے میں تھا کا کیا ہے ہے ہے
ہے ہے ہے ہے ہے ہیں گر...

  Temperature 1.0 (Sample 1 of 5):
  محبت سے مسائل میں قید نہیں اچھلے نہیں اپمان سدرہ ہیں
اپمان سدرہ ہیں بننے ہے طلب پکارنے ساحرؔ چراغان ہو آنکھیں ہے جھٹک
آنکھیں ہے جھٹک شیدا الست نہیں ہوں ہوا ہے گیا بست مسکراؤں ناتواں
بست مسکراؤں ناتواں...

  Temperature 1.3 (Sample 1 of 5):
  محبت صافیٔ خطرات تھا دلیر نایاب آرسی گہربار گوں کاج سے
گوں کاج سے روکوں مہربانیاں بتانا لائے رچے فگاروں لگاؤ گریے ٹوٹیں بگاڑ
گریے ٹوٹیں بگاڑ کھولا خوان نرالا معاذ گری تشویش ڈھیل پیر گریبانوں غاروں
پیر...

Generating poetry with seed: دل

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  دل محیط استقلال ہے ہیں ہے کر ہے سمجھاؤ ہیں کیا
س


Generating poetry with seed: محبت

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  محبت میں ہوگا یارب پر نہیں دھواں میں نے رکھا ہے
نے رکھا ہے طرح دم کیوں ہے شراب آخر ہوئے شوق والا کا
شوق والا کا کوئی کا طرف کو کا ہوگا عشق میں لیکن سکتے
میں لیکن سکتے ہیں ہم نشیں ہے پھری کر لینا ہم کو...

  Temperature 1.0 (Sample 1 of 5):
  محبت میں مار منتظر نہیں آتا ہے بولے ہے ابھی شمع
ہے ابھی شمع اور نہیں سکتا ہے پہنچے ہوئی پھول کبھی جاؤ تدبیریں
کبھی جاؤ تدبیریں کیجے میں ہوئی سے آہستہ آہستہ ہم مانگے ہے کھلا
مانگے ہے کھلا انگشت سب زخ...

  Temperature 1.3 (Sample 1 of 5):
  محبت کوئی پیامی رہا ہے دریا معلوم کھلا اسدؔ برملا رہیں
اسدؔ برملا رہیں ارماں میں ملتا کی ہوتی ہیں بہت کیا ہے کیا
کیا ہے کیا ہے ہوئی بار اب پلوائی ہوں میں مزاج سہی لگے
مزاج سہی لگے ہیں پوچھ بتاں ہوں گے...

Generating poetry with seed: دل

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  دل میں ہے میرے پاس ہوتا نہیں کیا نکلے گا کہلاوے



Generating poetry with seed: محبت

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  محبت ہے آگے پر میں ہے طرح بھی نہیں ہوتا ہے
نہیں ہوتا ہے ہم میں مہماں کی دوست گئیں پر مجھے کیا کیا
مجھے کیا کیا ہے لو میں مجھے کا لگے میں بھی نہیں رہا
بھی نہیں رہا ہے بیچ ہے واسطے دوست زلف مجھے گیا پر ...

  Temperature 1.0 (Sample 1 of 5):
  محبت ہے پوچھ کا دیجئے راز بھیگنے لکھنا سے سنو کریں
سے سنو کریں کہئے نیکی سی ہے آہستہ تیرا نے پایا کو کی
پایا کو کی ہے شراب کو کشتہ میں کوئی گیا کہاں کا حسابوں
کہاں کا حسابوں فساں اشارہ ور ہوئی شاید ہو...

  Temperature 1.3 (Sample 1 of 5):
  محبت بگاڑ اندھیری تمام الحمد جاے فوجداری سرگرم سوں وسیمؔ پر
سوں وسیمؔ پر بہار رنگوں اجڑی فقیری بدلا ہے اور اور ہو بدلا
اور ہو بدلا ہے لجاتے سخن پاسخ خواہ تھا مسل قراروں پایا باغ
قراروں پایا باغ کیا سر...

Generating poetry with seed: دل

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  دل میں تھے ہوتا ہے لینا آنکھیں رنگ بھی میں ہے
بھ


Generating poetry with seed: محبت

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  محبت تغافل آدھا نبھا ہنگامہ درویشی جاہ فسانا فراز نگاہیاں صییاد
فراز نگاہیاں صییاد طرف دوش ہاری زماں غصب لوث تیشۂ لگنے روکشی ترسا
لگنے روکشی ترسا کھاتی بندھیں رواج بلانے بجھانے شرماؤ حیف کارہ ستائی سل...

  Temperature 1.0 (Sample 1 of 5):
  محبت پھروں صفائی جمال مژگاں تخیلات امنگ لطمۂ لڑیے توجہ کفر
لڑیے توجہ کفر بہاتے چھپنا سج بیٹھا وفا شیوہ طرح خدایا چھاپ جینا
خدایا چھاپ جینا خوباں سنانی الجھتا پگھلائیں درماں مبتلائے صفائی تالے یاس وہ
ت...

  Temperature 1.3 (Sample 1 of 5):
  محبت گورنمنٹ بردار تماشہ استعارے عارض جوت حریصوں خاطر عنایت گیر
خاطر عنایت گیر پریشانیٔ پلٹتا بڑھے حکم سنورتا فق کرتا دوز دایم خرابہ
دوز دایم خرابہ اڑتا پانی، جانتے اٹھاتے گیتی شورشیں جامہ گداؤں دونوں...

Generating poetry with seed: دل

Sample generations (showing first sample per temperature):

  Temperature 0.7 (Sample 1 of 5):
  دل قائم امیدوں یادش پہچانتا سوغات نہ بڑھائی ٹالن

In [8]:
# Display all generated poems
for model_key, seed_poems in all_generated_poems.items():
    print(f"\n{'='*80}")
    print(f"Model: {model_key}")
    print(f"{'='*80}")

    for seed_word, temp_samples in seed_poems.items():
        print(f"\nSeed Word: {seed_word}")
        print("-" * 60)
        for temp in [0.7, 1.0, 1.3]:
            print(f"\nTemperature: {temp}")
            for i, sample in enumerate(temp_samples[temp][:2], 1):  # Show first 2 samples
                print(f"  Sample {i}:")
                print(f"  {sample[:150]}...")  # Truncated for display



Model: RNN_Adam

Seed Word: محبت
------------------------------------------------------------

Temperature: 0.7
  Sample 1:
  محبت سب راج دیکھیے دیا ہے مجھے مجھے اندیش قدم ہوگا
اندیش قدم ہوگا کیا نگار پر بھی ہیں ہم کو غم تھا مجھے
غم تھا مجھے رب آج اسے لیکن زلف لکھنؤ ہے ساقی م...
  Sample 2:
  محبت نزول تب نکلے گی ہم گزرے ہو جائے گا نکلا
جائے گا نکلا تھا میں لکھنؤ ہے آج عالم میں آوے ہے جوشؔ
آوے ہے جوشؔ بیتابی کی ہے پھری کدھر دیں گے کیا ہو کر...

Temperature: 1.0
  Sample 1:
  محبت میرا سحر کائی ہے مجھے آوارگی بوسہ ہستی آنکھیں عشق
ہستی آنکھیں عشق میرا تبرک کی راہ آشیانہ جالبؔ سزا شمار تھا کوئی
شمار تھا کوئی بالا پہچان آوے ہے...
  Sample 2:
  محبت سے تنہائی کا نازک شمار آرائی ہوا ہے سہل ہے
ہے سہل ہے لگے بھی ہے رسوم کیسی کہی آیا ہے مجھے حال
ہے مجھے حال رشک مست ہوگی جانے نمک بیچ غالبؔ تمہیں ...

Temperature: 1.3
  Sample 1:
  محبت چلتی جلوہ کرتے جاؤ سی نہیں سجتے پرکاری سامعین کدہ
پرکاری سامعین کدہ گوارا گلہ ناصحو چاہے ہے ہم کون اٹھائیے پاڑ جائے
اٹھائیے پاڑ جائے ہوگی قسم گھڑ...
  Sample 2:

## Save Generated Poems to Files


In [9]:
# Save generated poems to text files (all samples with all temperatures)
for model_key, seed_poems in all_generated_poems.items():
    filename = f"../results/generated_poems/{model_key.lower()}.txt"

    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"Generated Poetry - {model_key}\n")
        f.write("=" * 80 + "\n\n")

        for seed_word, temp_samples in seed_poems.items():
            f.write(f"Seed Word: {seed_word}\n")
            f.write("=" * 80 + "\n\n")

            for temp in [0.7, 1.0, 1.3]:
                f.write(f"Temperature: {temp}\n")
                f.write("-" * 60 + "\n")
                for i, sample in enumerate(temp_samples[temp], 1):
                    f.write(f"Sample {i}:\n")
                    f.write(f"{sample}\n")
                    f.write("\n")
                f.write("\n")

    print(f"Saved: {filename}")

print("\nAll generated poems saved successfully!")
print("Note: Each model-optimizer combination has 5 samples × 3 temperatures × 5 seeds = 75 total samples")


Saved: ../results/generated_poems/rnn_adam.txt
Saved: ../results/generated_poems/rnn_rmsprop.txt
Saved: ../results/generated_poems/rnn_sgd.txt
Saved: ../results/generated_poems/lstm_adam.txt
Saved: ../results/generated_poems/lstm_rmsprop.txt
Saved: ../results/generated_poems/lstm_sgd.txt

All generated poems saved successfully!
Note: Each model-optimizer combination has 5 samples × 3 temperatures × 5 seeds = 75 total samples
